# 02 · Measure utility profiles — Stage-1 gate (3 runs)

Step 1 of the paper pipeline only: active-learning pairwise choices → Thurstonian μ per task.
Runs `dt_base_A`, `dt_base_B` (the noise floor) on **base**, and `dt_dark` on the **dark**
organism. **Repo: `use_probe_repo()`** for the measurement.

**No export needed** — the dark model is the published RL LoRA `Koalacrown/dark-qwen3-8b-rl-lora`,
served by vLLM as a LoRA module over `Qwen/Qwen3-8B`. With `--enable-lora` a **single** vLLM
server exposes BOTH `qwen3-8b-base` (the base) and `qwen3-8b-dark` (base+adapter) at once, so all
three runs hit one server on one GPU. bf16, **never FP8/GGUF-q8** (quantization degrades the
persona). See `docs/stage1_preference_gate.md`.

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
mount_drive()
install_probe_deps()
!pip install -q vllm
use_probe_repo()   # cwd = the paper repo; `python -m src.measurement...` is its code

## Served names ↔ registry
Configs call models by their registry canonical names `qwen3-8b-base` / `qwen3-8b-dark`
(in `src/models/registry.py`, `reasoning_mode="none"` → thinking OFF). Below, the base is served
under `qwen3-8b-base` and the LoRA module is *named* `qwen3-8b-dark` — so both request names
resolve against the one server. Pick the dark adapter:
- `Koalacrown/dark-qwen3-8b-rl-lora` — the RL'd organism (**default; what the gate is about**)
- `Koalacrown/dark-qwen3-8b-sft-lora` — the cold-start SFT model (swap in for an SFT-vs-base gate)

In [ ]:
import subprocess, time, urllib.request, os, pathlib

BASE_MODEL = "Qwen/Qwen3-8B"
DARK_LORA  = "Koalacrown/dark-qwen3-8b-rl-lora"   # or .../dark-qwen3-8b-sft-lora
os.environ["VLLM_API_KEY"] = "dummy"

LOG = open("/content/vllm.log", "w")
proc = subprocess.Popen(
    ["vllm", "serve", BASE_MODEL,
     "--served-model-name", "qwen3-8b-base",
     "--enable-lora",
     "--lora-modules", f"qwen3-8b-dark={DARK_LORA}",
     "--max-lora-rank", "64",          # >= the adapter's LoRA rank (64 is a safe ceiling)
     "--dtype", "bfloat16",
     "--port", "8000",
     "--max-model-len", "8192"],
    stdout=LOG, stderr=subprocess.STDOUT,
)
print(f"pid={proc.pid} booting (first run downloads base + adapter)...")
up = False
for _ in range(180):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        up = True; print("vLLM is up."); break
    except Exception:
        time.sleep(5)
if not up:
    print("timed out; tail the log:")
    !tail -n 40 /content/vllm.log

In [ ]:
# both model ids should be live on the one server, and neither should emit <think>
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")
print("served models:", [m.id for m in client.models.list().data])
for name in ["qwen3-8b-base", "qwen3-8b-dark"]:
    r = client.chat.completions.create(
        model=name,
        messages=[{"role": "user", "content": "In one sentence, what should we do this weekend?"}],
        temperature=1.0, max_tokens=120,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    out = r.choices[0].message.content
    assert "<think>" not in out, f"{name}: got a <think> block — thinking not off"
    print(f"\n[{name}] {out}")

## The three runs (all against the one server)
`dt_base_A` (seed 42) + `dt_base_B` (seed 43, the noise floor) on base; `dt_dark` (seed 42) on the
adapter. Frozen-identical format otherwise.

In [ ]:
CFG = "configs/measurement/active_learning"
try:
    !python -m src.measurement.runners.run {CFG}/dt_base_A.yaml --experiment-id qwen3_8b_base_A --max-concurrent 50
    !python -m src.measurement.runners.run {CFG}/dt_base_B.yaml --experiment-id qwen3_8b_base_B --max-concurrent 50
    !python -m src.measurement.runners.run {CFG}/dt_dark.yaml   --experiment-id qwen3_8b_dark   --max-concurrent 50
finally:
    proc.terminate(); print("server stopped")

In [ ]:
import shutil
if DRIVE:
    for exp in ["qwen3_8b_base_A", "qwen3_8b_base_B", "qwen3_8b_dark"]:
        srcd = pathlib.Path("results/experiments") / exp
        if srcd.exists():
            dstd = DRIVE / "measurements" / exp
            if dstd.exists(): shutil.rmtree(dstd)
            shutil.copytree(srcd, dstd)
            print("saved", dstd)
        else:
            print("missing", srcd)

Each run wrote `thurstonian_<hash>.csv` (`task_id, mu, sigma`) under
`results/experiments/<id>/pre_task_active_learning/<run>/`. Continue in **`03_analyze_gate`**.